# Use transfer learning for large image classification, going through these steps:

### a. Create a training set containing at least 100 images per class. For example, you could classify your own pictures based on the location (beach, mountain, city,etc.). Alternatively, you can use an existing dataset, such as the one used in PyTorch’s transfer learning for computer vision tutorial.

### b. Split it into a training set, a validation set, and a test set.

### c. Build the input pipeline, apply the appropriate preprocessing operations, and optionally add data augmentation.

### d. Fine-tune a pretrained model on this dataset.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision.datasets import ImageFolder
import torchvision.transforms.v2 as transforms
import torchmetrics
from torch.utils.data import random_split, DataLoader
import torchvision
import PIL

In [2]:
data_transforms = {
    "train": transforms.Compose(
        [
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    ),
    "val": transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    ),
}

/opt/anaconda3/envs/ml/lib/python3.12/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [3]:
data_dir = "datasets/hymenoptera"

train_data = ImageFolder(root=f"{data_dir}/train", transform=data_transforms["train"])
print(type(train_data))
val_data = ImageFolder(root=f"{data_dir}/val", transform=data_transforms["val"])

<class 'torchvision.datasets.folder.ImageFolder'>


In [4]:
len(val_data)

153

In [ ]:
val_data, test_data = random_split(
    dataset=val_data, lengths=[70, 83], generator=torch.manual_seed(42)
)

In [6]:
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=16)
test_loader = DataLoader(dataset=test_data, batch_size=16)

In [7]:
device = "mps"

weights = torchvision.models.ConvNeXt_Small_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_small(weights=weights)

In [8]:
for name, child in model.named_children():
    print(name)

features
avgpool
classifier


In [9]:
print(model.classifier)

Sequential(
  (0): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True, bias=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=768, out_features=1000, bias=True)
)


In [10]:
model.classifier[2] = nn.Linear(768, 2)

In [11]:
for param in model.parameters():
    param.requires_grad = False

In [12]:
for param in model.classifier.parameters():
    param.requires_grad = True

In [13]:
optimizer = torch.optim.AdamW(model.parameters())
loss_fn = torch.nn.CrossEntropyLoss()
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.1
)
warm_up_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda epoch: (min(epoch, 3) / 3) * (1.0 - 0.1) + 0.1
)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(device)
model = model.to(device)

In [14]:
import time

device = "mps"


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train_with_early_stopping_lr_scheduler_warm_up(
    model,
    optimizer,
    loss_fn,
    metric,
    train_loader,
    valid_loader,
    n_epochs,
    warmup_scheduler,
    patience=10,
    checkpoint_path=None,
    scheduler=None,
):
    checkpoint_path = checkpoint_path or "my_checkpoint.pt"
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    best_metric = 0.0
    patience_counter = 0
    warmup_epochs = 3
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        t0 = time.time()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        train_metric = metric.compute().item()
        valid_metric = evaluate_tm(model, valid_loader, metric).item()
        if valid_metric > best_metric:
            torch.save(model.state_dict(), checkpoint_path)
            best_metric = valid_metric
            best = " (best)"
            patience_counter = 0
        else:
            patience_counter += 1
            best = ""

        t1 = time.time()
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)
        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train metric: {history['train_metrics'][-1]:.4f}, "
            f"valid metric: {history['valid_metrics'][-1]:.4f}{best}"
            f" in {t1 - t0:.1f}s"
        )
        if epoch < warmup_epochs:
            warmup_scheduler.step()
        if epoch >= warmup_epochs:
            if scheduler is not None:
                scheduler.step(valid_metric)
        if patience_counter >= patience:
            print("Early stopping!")
            break
    return history

In [15]:
results = train_with_early_stopping_lr_scheduler_warm_up(
    model,
    optimizer,
    loss_fn,
    metric,
    train_loader,
    val_loader,
    n_epochs=20,
    patience=5,
    warmup_scheduler=warm_up_scheduler,
    scheduler=perf_scheduler,
)

Epoch 1/20, train loss: 0.7265, train metric: 0.4303, valid metric: 0.5286 (best) in 6.8s
Epoch 2/20, train loss: 0.4970, train metric: 0.8975, valid metric: 1.0000 (best) in 6.4s
Epoch 3/20, train loss: 0.2346, train metric: 0.9713, valid metric: 1.0000 in 6.0s
Epoch 4/20, train loss: 0.1243, train metric: 0.9754, valid metric: 1.0000 in 6.0s
Epoch 5/20, train loss: 0.1068, train metric: 0.9795, valid metric: 1.0000 in 6.0s
Epoch 6/20, train loss: 0.0757, train metric: 0.9754, valid metric: 1.0000 in 6.0s
Epoch 7/20, train loss: 0.0606, train metric: 0.9836, valid metric: 1.0000 in 6.0s
Early stopping!


In [16]:
evaluate_tm(model, data_loader=test_loader, metric=metric)

tensor(1., device='mps:0')

In [17]:
for param in model.parameters():
    param.requires_grad = True


In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
loss_fn = torch.nn.CrossEntropyLoss()
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.1
)
warm_up_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda epoch: (min(epoch, 3) / 3) * (1.0 - 0.1) + 0.1
)
metric = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(device)

In [19]:
results = train_with_early_stopping_lr_scheduler_warm_up(
    model,
    optimizer,
    loss_fn,
    metric,
    train_loader,
    val_loader,
    n_epochs=20,
    patience=5,
    warmup_scheduler=warm_up_scheduler,
    scheduler=perf_scheduler,
)

Epoch 1/20, train loss: 0.0576, train metric: 0.9877, valid metric: 1.0000 (best) in 18.8s
Epoch 2/20, train loss: 0.0600, train metric: 0.9795, valid metric: 1.0000 in 18.2s
Epoch 3/20, train loss: 0.0518, train metric: 0.9877, valid metric: 1.0000 in 18.3s
Epoch 4/20, train loss: 0.0723, train metric: 0.9672, valid metric: 1.0000 in 18.3s
Epoch 5/20, train loss: 0.0342, train metric: 0.9959, valid metric: 1.0000 in 18.4s
Epoch 6/20, train loss: 0.0398, train metric: 1.0000, valid metric: 1.0000 in 18.4s
Early stopping!


In [20]:
evaluate_tm(model, data_loader=test_loader, metric=metric)

tensor(1., device='mps:0')